# PoliMillionaire — NLP Chatbot Assignment

A chatbot that plays the *Who wants to be a PoliMillionaire?* quiz via a REST API client.

**Competitions:**
| ID | Name | Questions |
|----|------|-----------|
| 0  | Entertainment | 15 |
| 1  | Ancient History & Politics | 15 |
| 2  | Science & Nature | 15 |
| 3  | Maths | 15 |

**Architecture:**
- **LLM:** Qwen2.5-7B-Instruct (float16, greedy decoding for speed)
- **RAG:** DuckDuckGo/ddgs (Entertainment) · Wikipedia (History) · Multi-query Wikipedia+DuckDuckGo + cross-encoder reranking (Science) · Calculator (Maths)
- **Answer extraction:** regex-based, robust fallback chain (`ANSWER: X` tag → digit → letter)

## 0 · Setup — Clone from GitHub

Clone the repository so `millionaire_client` and `millionaire_bot.py` are available.

In [1]:
!git clone https://github.com/riccardo03/NLP_university_project.git 2>/dev/null || (cd NLP_university_project && git pull)

In [2]:
import sys, os

PACKAGE_DIR = '/content/NLP_university_project'

if PACKAGE_DIR not in sys.path:
    sys.path.append(PACKAGE_DIR)

print('Path configured:', PACKAGE_DIR)

Path configured: /content/NLP_university_project


## 1 · Install dependencies

Required packages beyond the standard Colab environment:

In [3]:
%pip install -q ddgs wikipedia transformers accelerate sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 82.9 MB/s eta 0:00:00


## 2 · Import the bot module

All game logic lives in `millionaire_bot.py`. Import it, we do.

In [4]:
import millionaire_bot as bot

print('Imported successfully, the bot module has.')

Imported successfully, the bot module has.


## 3 · Load the LLM

Qwen2.5-7B-Instruct is loaded with `device_map="auto"` and `torch_dtype=float16`.
This downloads ~14 GB on first run — patience, the Force requires.

In [5]:
bot.load_model('Qwen/Qwen2.5-7B-Instruct')

Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

  [RAG-Science] Cross-encoder ready, it is.
  [Warmup] All models ready.


In [6]:
import subprocess, importlib

# Clean the repository before pulling to avoid conflicts
print("Cleaning local repository (discarding local changes and untracked files)...")
subprocess.run(["git", "-C", "/content/NLP_university_project", "reset", "--hard"], check=True)
subprocess.run(["git", "-C", "/content/NLP_university_project", "clean", "-fdx"], check=True)
print("Pulling latest changes from GitHub...")
subprocess.run(["git", "-C", "/content/NLP_university_project", "pull"], check=True)

# Salva il modello e la pipeline già caricati
_model    = bot._model
_pipe     = bot._pipe
_tokenizer = bot._tokenizer

# Ricarica il modulo
importlib.reload(bot)

# Reinietta i riferimenti — così load_model() non viene rieseguito
bot._model    = _model
bot._pipe     = _pipe
bot._tokenizer = _tokenizer

print("Modulo aggiornato, modello preservato.")

Cleaning local repository (discarding local changes and untracked files)...
Pulling latest changes from GitHub...
Modulo aggiornato, modello preservato.


## 4 · Connect to the API

Authenticate with the PoliMillionaire server. Store your password in a Colab secret named `poli-millionaire`.

In [7]:
from google.colab import userdata
from millionaire_client import MillionaireClient, AuthenticationError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'riccardo'           # Change to your username
PASSWORD = userdata.get('poli-millionaire')

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Welcome, {user.username}! Role: {user.role}')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Welcome, riccardo! Role: student


## 5 · Browse available competitions

In [8]:
competitions = client.competitions.list_all()
print('=== Available Competitions ===')
for c in competitions:
    print(f'  [{c.id}] {c.name} — {c.max_levels} questions')

=== Available Competitions ===
  [0] Entertainment — 15 questions
  [1] Ancient History and Politics — 15 questions
  [2] Science and Nature — 15 questions
  [3] Maths — 15 questions


## 6 · Play a single competition

Choose a `COMP_ID` (0–3) and let the bot play one full game.

In [12]:
COMP_ID = bot.COMP_ENTERTAINMENT   # Change this to try other competitions

game = client.game.start(competition_id=COMP_ID)
print(f'Session ID: {game.session_id} | Questions: {game.state.competition.max_levels}')

game_log = bot.play_game(game, COMP_ID)
bot.print_evaluation(game_log)

Session ID: 88269 | Questions: 15

  Starting: Entertainment

--- Level 1 | Time: 29.9s ---
Q: How does the blues form relate to the 12-bar blues structure?
  [0] It is a superset
  [1] It is unrelated
  [2] It is an alternative form
  [3] It is a subset
  [RAG] Searching for context...
  [RAG-Entertainment] Query: '12 bar blues structure blues form relation'
  [RAG] Done in 8.0s. Context: The twelve-bar blues (or blues changes) is one of the most prominent chord progressions in popular music. The blues prog...
  [LLM] Thinking...
  [LLM] Output: '3' → Answer ID: 3 (in 2.1s)
  ✗ WRONG! Game over. Earned: $0.00

  Entertainment — Level reached: 1 | Earnings: $0.00


──────────────────────────────────────────────────
  EVALUATION — Entertainment
──────────────────────────────────────────────────
  Level reached : 1
  Earnings      : $0.00
  Questions     : 1
  Correct       : 0
  Timed out     : 0
  Accuracy      : 0.0%
──────────────────────────────────────────────────
  [✗] L1: How doe

## 7 · Play all four competitions

Run all competitions sequentially and aggregate results.
A complete evaluation, this produces.

In [ ]:
all_logs = []

for comp_id in [bot.COMP_ENTERTAINMENT,
                bot.COMP_HISTORY_POLITICS,
                bot.COMP_SCIENCE_NATURE,
                bot.COMP_MATHS]:
    game = client.game.start(competition_id=comp_id)
    print(f'Session {game.session_id} started for competition {comp_id}.')
    log = bot.play_game(game, comp_id)
    all_logs.append(log)

bot.print_all_evaluations(all_logs)

Session 58039 started for competition 0.

  Starting: Entertainment

--- Level 1 | Time: 29.9s ---
Q: What was the primary reason James Cameron switched from studying physics to English at Fullerton College?
  [0] He wanted to become a writer
  [1] He was inspired by the success of Star Wars
  [2] He found physics too difficult
  [3] He was offered a job in the film industry
  [RAG] Searching for context...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG] Done in 1.6s. Context: [James Cameron - Wikipedia] 1 day ago - After high school, Cameron enrolled at Fullerton College, a community college, i...
  [LLM] Thinking...
  [LLM] Output: '0' → Answer ID: 0 (in 1.8s)
  ✗ WRONG! Game over. Earned: $0.00

  Entertainment — Level reached: 1 | Earnings: $0.00

Session 58040 started for competition 1.

  Starting: Ancient History & Politics

--- Level 1 | Time: 29.9s ---
Q: How many ancient biographies of Homer are mentioned in the article?
  [0] One
  [1] Four
  [2] Three
  [3] Two
  [RAG] Searching for context...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG] Done in 1.2s. Context: Ancient accounts of Homer include numerous passages in which archaic and classical Greek poets and prose authors mention...
  [LLM] Thinking...
  [LLM] Output: '2' → Answer ID: 2 (in 1.6s)
  ✗ WRONG! Game over. Earned: $0.00

  Ancient History & Politics — Level reached: 1 | Earnings: $0.00



...

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG-Science] Stage2 retrieval: 3.6s → 5 snippets
  [RAG-Science] Stage3 reranking: 0.0s
  [RAG] Done in 19.1s. Context: [Soil erosion - Wikipedia] Valley or stream erosion occurs with continued water flow along a linear feature. The erosion...
  [LLM] Thinking...
  [LLM] Output: '1' → Answer ID: 1 (in 2.1s)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ CORRECT! Earned so far: $100.00

--- Level 2 | Time: 29.9s ---
Q: Which condition is necessary for metamorphic rocks to form?
  [0] slow cooling of magma
  [1] constant weathering and erosion
  [2] extreme pressure and heating
  [3] rapid burial of sediments
  [RAG] Searching for context...


KeyboardInterrupt: 

## 8 · Leaderboard

See where the bot ranks after playing.

In [ ]:
for comp_id in range(4):
    lb = client.leaderboard.get(competition_id=comp_id, limit=10)
    print(f'\n=== Leaderboard: {lb.competition.name} ===')
    for i, entry in enumerate(lb.entries[:10], 1):
        marker = ' <-- YOU' if entry.username == USERNAME else ''
        print(f'  {i:>2}. {entry.username:<20} ${entry.score:>10,.2f}  (Level {entry.reached_level}){marker}')

## 9 · Quick unit tests for key functions

Verify the helper functions work correctly before running a live game.

In [ ]:
# --- Test extract_answer_id ---
cases = [
    ('The answer is 2, definitely.',          0, 4, 2),
    ('I think option B is correct.',          0, 4, 1),
    ('No clue at all.',                       0, 4, 0),
    ('3_Full chain-of-thought reasoning...',  0, 4, 3),
    ('Option A seems right.',                 0, 4, 0),
]
print('=== extract_answer_id tests ===')
all_passed = True
for text, default, n_opts, expected in cases:
    got = bot.extract_answer_id(text, n_opts)
    status = '✓' if got == expected else '✗'
    if got != expected:
        all_passed = False
    print(f'  {status}  "{text[:40]}" → {got} (expected {expected})')
print('All passed!' if all_passed else 'Some failed!')

# --- Test rag_maths ---
print('\n=== rag_maths tests ===')
maths_cases = [
    'What is 15% of 200?',
    'Calculate the square root of 144.',
    'What is 2^10?',
    'What is 3 + 4 * 2?',
]
for q in maths_cases:
    result = bot.rag_maths(q)
    print(f'  Q: {q}')
    print(f'  → {result}\n')

## 10 · Save logs to JSON

Persist all game logs for offline analysis.

In [ ]:
import json

log_path = '/content/game_logs.json'
with open(log_path, 'w') as f:
    json.dump(all_logs, f, indent=2)

print(f'Saved {len(all_logs)} game log(s) to {log_path}')